### Simple Gen AI APP using LangChain

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPEN_AI_KEY']=os.getenv("OPEN_AI_KEY")
os.environ["USER_AGENT"] = "my-langchain-app"

## Langsmith Tracking Keys, 3 keys
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT")

In [34]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [54]:
## Data Ingestion--From the website we need to scrape the data
loader = WebBaseLoader("https://docs.langchain.com/langsmith/administration-overview")
docs = loader.load()
# docs

In [55]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)
# documents

In [21]:
# dcouments-->text-->vectors-->Vector Embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
embedded_docs = embeddings.embed_documents([d.page_content for d in documents])
embedded_docs

[[-0.028115213,
  0.018532984,
  -0.16841224,
  -0.059171807,
  -0.0006458957,
  -0.08089028,
  -0.04603127,
  -0.0008114351,
  -0.048227057,
  -0.010433661,
  -0.0066590267,
  0.03324879,
  0.10043482,
  -0.0014959832,
  0.038569033,
  -0.033588078,
  -0.0672347,
  -0.029683221,
  -0.008838935,
  0.020632006,
  -0.031067409,
  0.044323493,
  -0.034372374,
  -0.027270764,
  0.08336887,
  0.036162734,
  0.0007611073,
  0.0037924398,
  -0.065501556,
  -0.03565315,
  0.04467993,
  -0.013620672,
  -0.031300925,
  0.005334819,
  -0.069325045,
  -0.054126173,
  0.059699826,
  -0.02423644,
  -0.028563334,
  0.08975778,
  0.035093114,
  0.0074316896,
  0.0037602156,
  -0.013120779,
  0.03209731,
  -0.010674713,
  -0.022351941,
  0.01051834,
  0.040679965,
  -0.05878276,
  -0.011152689,
  -0.0045370935,
  -0.041739564,
  0.053587824,
  0.05013694,
  0.018203786,
  0.009387133,
  0.0077678366,
  0.00014550498,
  -0.062889375,
  0.094771326,
  0.06144957,
  -0.027224446,
  0.049240302,
  0.030004

In [56]:
# Vector Store DB
vectorstoredb=FAISS.from_documents(documents,embeddings)
# vectorstoredb

In [38]:
## Query From a vector db
query="What is organization?"
# query = 'How many workspaces can an organization have?'
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Organizations\nAn organization is a logical grouping of users within LangSmith that defines shared settings applying across all of its workspaces. These settings govern organization-wide concerns rather than the agents and resources within a workspace. Common organization-level configurations include user management, single sign-on (SSO), OAuth provider configuration, custom role creation, billing, and usage tracking.\nTypically, there is one organization per company. An organization can have multiple workspaces. For configuration details, see Set up an organization.\nWhen you log in for the first time, a personal organization will be created for you automatically. If you’d like to collaborate with others, you can create a separate organization and invite your team members to join. There are a few important differences between your personal organization and shared organizations:'

In [27]:
llm = ChatOllama(
    model="gpt-oss:20b",   # exactly as shown in `ollama list`
    temperature=0.7,
    # base_url="http://localhost:11434",  # only needed if Ollama runs elsewhere
)

In [64]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>

Question: {input}
"""
)
document_chain=create_stuff_documents_chain(llm,prompt)
# document_chain

from langchain_core.documents import Document
document_chain.invoke({
    "input":"what is love?",
    "context":[Document(page_content="LOVE IS AWESOME LOLO, and makes the world a better place")]
})

'Love is awesome and it makes the world a better place.'

In [58]:
# However, we want the documents to first come from the retriever we just set up. That way, 
# we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.
from langchain_classic.chains import create_retrieval_chain

retriever = vectorstoredb.as_retriever(search_kwargs={"k": 3})
retrieval_chain = create_retrieval_chain(retriever, document_chain)

response = retrieval_chain.invoke({"input": "What is organization?"})
# print(response["context"])
print(response["answer"])

An **organization** in LangSmith is a logical grouping of users that represents a company or a shared team. It sets up organization‑wide settings—such as user management, single sign‑on (SSO), OAuth provider configuration, custom role creation, billing, and usage tracking—that apply to every workspace inside that organization. Typically, there is one organization per company, but an organization can contain multiple workspaces. When you first log in, a personal organization is created automatically; you can then create a separate shared organization and invite teammates to collaborate.
